# 1. Almacenamiento y consulta de ventas

## Contexto

Voy a simular una tienda especializada en videojuegos, consolas y accesorios, con registros de ventas correspondientes a septiembre, octubre y noviembre de 2026.

El objetivo es almacenar los datos en una tabla Delta utilizando Unity Catalog, para posteriormente realizar procesos de limpieza, transformación y automatización.

Los datos son ficticios y se incluyen intencionalmente tres problemas de calidad:

- Una venta duplicada.
- Una venta sin producto.
- Una venta con cantidad negativa.

Los precios están expresados en pesos colombianos (COP). GTA VI se registra como una preventa simulada.


## 1.1. Preguntas de investigación

### ¿Qué es Unity Catalog en Databricks?

Unity Catalog es una herramienta que sirve para organizar y gestionar los datos que tenemos en Databricks. Por ejemplo, permite crear catálogos, esquemas y tablas, además de controlar qué usuarios pueden acceder a la información. También facilita trabajar con los mismos datos desde diferentes notebooks.

### ¿Qué diferencia hay entre un DataFrame de Spark y una tabla Delta?

Un DataFrame de Spark es una estructura que nos permite trabajar con datos, ya sea para filtrarlos, consultarlos o realizar diferentes transformaciones.

Ya hablando de las tablas Delta, estas permiten almacenar los datos de forma permanente. Además, cuentan con funciones como las transacciones ACID y el historial de versiones, lo que ayuda a mantener la información organizada y evitar problemas al modificar los datos.

### Fuentes consultadas

- [Unity Catalog - Databricks](https://docs.databricks.com/aws/en/data-governance/unity-catalog/)
- [Delta Lake - Databricks](https://docs.databricks.com/aws/en/delta/)
- [Apache Spark SQL y DataFrames](https://spark.apache.org/docs/latest/sql-programming-guide.html)



In [0]:

from pyspark.sql.types import (
    StructType, StructField, IntegerType,
    StringType, DateType, DoubleType
)
from datetime import date

schema = StructType([
    StructField("id_venta", IntegerType(), False),
    StructField("fecha", DateType(), True),
    StructField("producto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio_unitario", DoubleType(), True)
])

ventas = [
    (1, date(2026, 9, 3), "The Legend of Zelda: Link's Awakening Remake", "Videojuegos", 2, 249900.0),
    (2, date(2026, 9, 8), "Minecraft", "Videojuegos", 4, 119900.0),
    (3, date(2026, 9, 15), "Control Xbox Series", "Accesorios", 3, 289900.0),
    (4, date(2026, 9, 22), "PlayStation 5", "Consolas", 1, 2599900.0),
    (5, date(2026, 10, 2), "GTA VI - Preventa", "Preventas", 2, 349900.0),
    (6, date(2026, 10, 6), "Audifonos HyperX Cloud II", "Accesorios", 3, 329900.0),
    (7, date(2026, 10, 10), "Nintendo Switch 2", "Consolas", 2, 2199900.0),
    (8, date(2026, 10, 14), "EA Sports FC 26", "Videojuegos", 3, 299900.0),
    (9, date(2026, 10, 20), "Teclado mecanico Redragon", "Accesorios", 4, 189900.0),
    (10, date(2026, 10, 28), "Control DualSense PS5", "Accesorios", 2, 329900.0),
    (11, date(2026, 11, 2), "Super Mario Bros. Wonder", "Videojuegos", 2, 259900.0),
    (12, date(2026, 11, 5), "Mouse Logitech G502", "Accesorios", 3, 249900.0),
    (13, date(2026, 11, 8), None, "Accesorios", 2, 99900.0),
    (14, date(2026, 11, 10), "Monitor Gamer 144Hz", "Accesorios", -1, 899900.0),
    (3, date(2026, 9, 15), "Control Xbox Series", "Accesorios", 3, 289900.0)
]

df = spark.createDataFrame(ventas, schema)

display(df)
print("Total de registros:", df.count())


id_venta,fecha,producto,categoria,cantidad,precio_unitario
1,2026-09-03,The Legend of Zelda: Link's Awakening Remake,Videojuegos,2,249900.0
2,2026-09-08,Minecraft,Videojuegos,4,119900.0
3,2026-09-15,Control Xbox Series,Accesorios,3,289900.0
4,2026-09-22,PlayStation 5,Consolas,1,2599900.0
5,2026-10-02,GTA VI - Preventa,Preventas,2,349900.0
6,2026-10-06,Audifonos HyperX Cloud II,Accesorios,3,329900.0
7,2026-10-10,Nintendo Switch 2,Consolas,2,2199900.0
8,2026-10-14,EA Sports FC 26,Videojuegos,3,299900.0
9,2026-10-20,Teclado mecanico Redragon,Accesorios,4,189900.0
10,2026-10-28,Control DualSense PS5,Accesorios,2,329900.0


Total de registros: 15


## 1.3. Almacenamiento en una tabla Delta

Una vez creado el DataFrame, se almacena en Unity Catalog utilizando el formato Delta.

La tabla se guardará con la siguiente estructura:

- **Catálogo:** workspace
- **Esquema:** ingenieria_datos_luismiguel
- **Tabla:** ventas_bronze

Se utiliza el modo `overwrite` para reemplazar el contenido de la tabla si se ejecuta nuevamente la carga, evitando acumular registros duplicados por ejecuciones repetidas.


In [0]:

tabla_bronze = "workspace.ingenieria_datos_luismiguel.ventas_bronze"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(tabla_bronze)

print("Tabla Delta creada correctamente")
print("Ubicación:", tabla_bronze)
print("Registros guardados:", spark.table(tabla_bronze).count())


Tabla Delta creada correctamente
Ubicación: workspace.ingenieria_datos_luismiguel.ventas_bronze
Registros guardados: 15


## 1.4. Verificación y consulta de ventas

Se consulta la tabla Delta almacenada para verificar que los registros se guardaron correctamente.

Se realizan las siguientes operaciones:

1. Consultar la cantidad total de registros.
2. Identificar las columnas y sus tipos de datos.
3. Filtrar las ventas de la categoría Videojuegos.

Esto permite comprobar que los datos se pueden recuperar desde Unity Catalog.


In [0]:

ventas_bronze = spark.table(tabla_bronze)

print("Total de registros:", ventas_bronze.count())

print("Columnas y tipos de datos:")
ventas_bronze.printSchema()

print("Ventas de la categoría Videojuegos:")
display(
    ventas_bronze
    .filter(ventas_bronze.categoria == "Videojuegos")
    .orderBy("fecha")
)


Total de registros: 15
Columnas y tipos de datos:
root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)

Ventas de la categoría Videojuegos:


id_venta,fecha,producto,categoria,cantidad,precio_unitario
1,2026-09-03,The Legend of Zelda: Link's Awakening Remake,Videojuegos,2,249900.0
2,2026-09-08,Minecraft,Videojuegos,4,119900.0
8,2026-10-14,EA Sports FC 26,Videojuegos,3,299900.0
11,2026-11-02,Super Mario Bros. Wonder,Videojuegos,2,259900.0


## 1.5. Conclusión

Se creó un DataFrame de Spark con 15 registros de ventas simuladas y se almacenó en una tabla Delta llamada `ventas_bronze` dentro de Unity Catalog.

La tabla permite conservar los registros y consultarlos posteriormente desde otros notebooks.

También se mantuvieron los errores intencionales en los datos originales, ya que serán utilizados en el siguiente ejercicio para aplicar reglas de limpieza y calidad.
